In [5]:
import arcpy
import os

def clip_rasters_from_mask(mask_layer, output_folder):
    """
    Clips rasters to mask feature boundaries based on filepath attribute.
    Handles coordinate system mismatches by projecting the clipping geometry.
    """
    
    # Clean up environment
    arcpy.env.overwriteOutput = True
    arcpy.env.addOutputsToMap = False # Prevents temp layers from clogging the TOC
    
    try:
        if not arcpy.Exists(mask_layer):
            arcpy.AddError(f"Mask layer '{mask_layer}' does not exist.")
            return
        
        if not os.path.exists(output_folder):
            os.makedirs(output_folder)
            arcpy.AddMessage(f"Created output folder: {output_folder}")
        
        # Handle selection
        desc = arcpy.Describe(mask_layer)
        if desc.FIDSet:
            count = int(arcpy.management.GetCount(mask_layer).getOutput(0))
            arcpy.AddMessage(f"Processing {count} selected features")
        else:
            count = int(arcpy.management.GetCount(mask_layer).getOutput(0))
            arcpy.AddMessage(f"Processing all {count} features")
        
        processed = 0
        errors = 0
        
        # We need the spatial reference of the mask layer itself
        mask_sr = desc.spatialReference
        
        # Iterate
        with arcpy.da.SearchCursor(mask_layer, ["SHAPE@", "filepath", "OID@"]) as cursor:
            for row in cursor:
                feature_geom = row[0]
                raster_filepath = row[1]
                oid = row[2]
                
                try:
                    # 1. Validation
                    if not raster_filepath or not os.path.exists(raster_filepath):
                        arcpy.AddWarning(f"OID {oid}: Invalid or missing file path: {raster_filepath}")
                        continue
                        
                    # 2. Path handling
                    path_parts = os.path.normpath(raster_filepath).split(os.sep)
                    subfolder = path_parts[-2] if len(path_parts) >= 2 else ""
                    filename = path_parts[-1]
                    
                    out_sub = os.path.join(output_folder, subfolder) if subfolder else output_folder
                    if not os.path.exists(out_sub):
                        os.makedirs(out_sub)
                        
                    out_raster = os.path.join(out_sub, filename)
                    
                    # 3. Spatial Reference Check & Geometry Projection
                    # We must project the feature geometry to match the RASTER's coordinate system
                    raster_desc = arcpy.Describe(raster_filepath)
                    raster_sr = raster_desc.spatialReference
                    
                    # Project geometry if needed
                    if mask_sr.name != raster_sr.name:
                        # arcpy.AddMessage(f"Reprojecting geometry for OID {oid} to match raster...")
                        proj_geom = feature_geom.projectAs(raster_sr)
                    else:
                        proj_geom = feature_geom
                    
                    # 4. Prepare Clipping Geometry (Temp Feature Class)
                    # Clip tool needs a feature class input for the clipping_geometry option
                    temp_fc = r"in_memory\clip_geom_{}".format(oid)
                    
                    # Create a temp feature class with the RASTER's spatial reference
                    arcpy.management.CreateFeatureclass("in_memory", f"clip_geom_{oid}", "POLYGON", spatial_reference=raster_sr)
                    
                    # Insert the (projected) geometry
                    with arcpy.da.InsertCursor(temp_fc, ["SHAPE@"]) as ins_cursor:
                        ins_cursor.insertRow([proj_geom])
                    
                    # 5. Calculate Extent (Rectangle) from PROJECTED geometry
                    # The rectangle strings must be in the raster's units
                    extent = proj_geom.extent
                    rect = f"{extent.XMin} {extent.YMin} {extent.XMax} {extent.YMax}"
                    
                    # 6. Execute Clip
                    arcpy.AddMessage(f"Clipping {filename}...")
                    
                    arcpy.management.Clip(
                        in_raster=raster_filepath,
                        rectangle=rect,
                        out_raster=out_raster,
                        in_template_dataset=temp_fc,
                        nodata_value="256", # Or leave empty "#"
                        clipping_geometry="ClippingGeometry",
                        maintain_clipping_extent="NO_MAINTAIN_EXTENT"
                    )
                    
                    # Check if output actually exists and has size
                    if arcpy.Exists(out_raster):
                        processed += 1
                        arcpy.AddMessage(f"Success: {out_raster}")
                    else:
                        arcpy.AddWarning(f"OID {oid}: Clip ran but output not found.")
                    
                    # Cleanup specific temp layer
                    arcpy.management.Delete(temp_fc)
                    
                except Exception as e:
                    errors += 1
                    arcpy.AddError(f"Error on OID {oid}: {str(e)}")
                    # Try to clean up temp layer if it failed mid-way
                    if arcpy.Exists(f"in_memory\\clip_geom_{oid}"):
                         arcpy.management.Delete(f"in_memory\\clip_geom_{oid}")
                    continue

        arcpy.AddMessage(f"Finished. Processed: {processed}, Errors: {errors}")
        
    except Exception as e:
        arcpy.AddError(f"Fatal Error: {str(e)}")


In [4]:
# For ArcGIS Notebook (manual entry):
mask_layer = "pentaria_masks_updated"  # Replace with your layer name from Contents
output_folder = r"C:\workspace\Freelance life\Pantelis Karapatsios\ArcPy ortho automation\Data\Structured folders\5ARIA_GEO_CORRECT_Clipped"  # Replace with your desired output folder

# Run the function
clip_rasters_from_mask(mask_layer, output_folder)